In [95]:
import re
from pathlib import Path
import numpy as np
from PIL import Image

# ==========================================
# НАСТРОЙКИ ПУТЕЙ
# ==========================================
# Корневая папка лабораторной работы
BASE_DIR = Path(r"C:\Users\RedBrean\Documents\MIPT\term2\MQO\experiment_folder")
INPUT_DIR = Path(r"C:\Users\RedBrean\Pictures\RQC\26_06_2026")
# INPUT_DIR = BASE_DIR / "raw_data"
OUTPUT_DIR = BASE_DIR / "processed"


# Создаем выходную директорию, если её нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# ФУНКЦИИ ОБРАБОТКИ ИЗОБРАЖЕНИЙ
# ==========================================

def load_image(filepath: Path) -> np.ndarray:
    """Загружает PNG изображение как float64 массив."""
    with Image.open(filepath) as img:
        return np.array(img, dtype=np.float64)


def combine_images(filepaths: list[Path], method: str = "median") -> np.ndarray:
    """
    Аккуратно объединяет стек изображений.
    'median' — убирает случайные шумы и горячие пиксели (рекомендуется).
    'mean'   — классическое усреднение.
    """
    if not filepaths:
        raise ValueError("Список файлов для объединения пуст.")
    
    stack = np.stack([load_image(f) for f in filepaths])
    
    if method == "median":
        return np.median(stack, axis=0)
    return np.mean(stack, axis=0)


def save_image(arr: np.ndarray, filepath: Path) -> None:
    """Сохраняет массив как PNG, автоматически определяя разрядность (8/16 бит)."""
    # Обрезаем значения, чтобы не вылететь за границы диапазона
    arr_clipped = np.clip(arr, 0, 65535)

    # Если исходный макс был небольшой, сохраняем в 8 бит, иначе в 16 бит
    if arr_clipped.max() <= 255:
        arr_rescaled = arr_clipped.astype(np.uint8)
    else:
        arr_rescaled = arr_clipped.astype(np.uint16)

    Image.fromarray(arr_rescaled).save(filepath)


# ==========================================
# ГРУППИРОВКА ФАЙЛОВ
# ==========================================

groups = {}

# Регулярное выражение для поиска даты/времени в названии
timestamp_pattern = re.compile(r"(.+?)_\d{4}-\d{2}-\d{2}T")

for file in INPUT_DIR.glob("*.png"):
    name = file.stem

    if "_NOISE_" in name:
        key = name.split("_NOISE_")[0]
        kind = "noise"
    else:
        match = timestamp_pattern.match(name)
        key = match.group(1) if match else "_".join(name.split("_")[:2])
        kind = "signal"

    groups.setdefault(key, {"signal": [], "noise": []})
    groups[key][kind].append(file)

# ==========================================
# ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ
# ==========================================

print(f"Старт обработки файлов из: {INPUT_DIR}\n" + "-"*50)

for key, data in sorted(groups.items()):
    signal_files = data["signal"]
    noise_files = data["noise"]

    if not signal_files:
        print(f"Предупреждение: Для группы '{key}' нет файлов сигнала. Пропускаем.")
        continue

    print(f"Обработка группы: {key}")
    print(f"  └─ Сигнальных кадров: {len(signal_files)}")
    print(f"  └─ Шумовых кадров (dark frames): {len(noise_files)}")

    # Усредняем медианой для защиты от выбросов
    signal_avg = combine_images(signal_files, method="mean")

    if noise_files:
        noise_avg = combine_images(noise_files, method="mean")
        # Вычитание темнового кадра (шума)
        result = signal_avg - noise_avg
    else:
        result = signal_avg

    # Убираем отрицательные пиксели, которые могли возникнуть при вычитании шума
    result = np.maximum(result, 0)

    # Сохраняем результат
    output_path = OUTPUT_DIR / f"{key}.png"
    save_image(result, output_path)

print("-"*50)
print("Готово! Чистые данные лежат здесь:")
print(OUTPUT_DIR)

Старт обработки файлов из: C:\Users\RedBrean\Pictures\RQC\26_06_2026
--------------------------------------------------
Обработка группы: 104_1_5_FILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 5
Обработка группы: 104_1_5_NOFILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 5
Обработка группы: 104_1_8_FILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 5
Обработка группы: 104_1_8_NOFILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 5
Обработка группы: 104_1_FILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 6
Обработка группы: 104_1_NOFILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 5
Обработка группы: 104_2_5_FILTER
  └─ Сигнальных кадров: 6
  └─ Шумовых кадров (dark frames): 0
Обработка группы: 104_2_5_NOFILTER
  └─ Сигнальных кадров: 7
  └─ Шумовых кадров (dark frames): 6
Обработка группы: 104_2_FILTER
  └─ Сигнальных кадров: 5
  └─ Шумовых кадров (dark frames): 5
Обработка груп

In [97]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns  # Удобно для красивых графиков с группировкой
from scipy.optimize import curve_fit
from scipy.signal import find_peaks, savgol_filter
from sklearn.metrics import r2_score


class CrystalExperiment:
    """
    Класс отдельного эксперимента.
    Хранит метаданные в виде произвольного словаря, осуществляет калибровку
    и анализ интерференционной картины. Все результаты сохраняются в поля класса.
    """
    C_LIGHT = 299_792_458

    def __init__(self, filter_filename, nofilter_filename, metadata=None, config=None):
        self.filter_filename = filter_filename
        self.nofilter_filename = nofilter_filename
        
        # Хранилище для любых параметров эксперимента (дистанция, щель, ток и т.д.)
        self.metadata = metadata if metadata is not None else {}
        
        if config is None:
            config = {
                "left": 250, "right": 370,
                "top": 140, "bottom": 275,
                "target_wavelength": 808.0,
                "output_dir": "analysis_output",
                "stripe_width": 3,
                "savgol_window": 13,
                "savgol_poly": 2,
                "peak_prominence": 0.008,
                "valley_prominence": 0.008,
                "peak_distance": 6,
                "peak_height_min": 0.12
            }
        self.config = config

        self.filter_dir = os.path.join(config["output_dir"], "filter_fits")
        self.nofilter_dir = os.path.join(config["output_dir"], "nofilter_spectra")

        # --- ПОЛЯ КЛАССА ДЛЯ ХРАНЕНИЯ РЕЗУЛЬТАТОВ ---
        # Результаты калибровки (Fit)
        self.alpha = None
        self.beta = None
        self.shift = None
        self.scale = None
        self.r2 = None
        
        # Массивы данных
        self.img_nf = None
        self.crop_nf = None
        self.spectrum_nf = None
        self.slice_808 = None
        self.smoothed_slice = None
        
        # Экстремумы и координаты
        self.pixel808 = None
        self.peaks_idx = []
        self.valleys_idx = []
        self.paired_extrema = []  
        self.local_visibilities = []

        # Ключевые физические метрики
        self.visibility = 0.0
        self.intensity_max_slice = 0.0  
        self.intensity_808_exact = 0.0
        self.intensity_808_avg = 0.0
        self.intensity_total = 0.0
        
        self.log_line = ""

    @staticmethod
    def _gaussian(w, w0, sig):
        return np.exp(-0.5 * ((w - w0) / sig) ** 2)

    @staticmethod
    def _convert_w_to_l(func, w_range, *params):
        l_range = 2 * np.pi * CrystalExperiment.C_LIGHT / w_range * 1e-3
        y_w = func(w_range, *params)
        y_l = y_w / l_range**2
        y_l /= np.max(y_l)
        return y_l

    @staticmethod
    def _gfit(pix, alpha, beta, shift, scale):
        wavelength = shift + scale * pix
        c_factor = 2 * np.pi * CrystalExperiment.C_LIGHT * 1e-3
        ang_freq = c_factor / wavelength
        l0 = 810
        l_fwhm = 10
        w0 = c_factor / l0
        w_fwhm = c_factor * l_fwhm / (l0**2 - (l_fwhm / 2)**2)
        sig_w = w_fwhm / (2 * np.sqrt(2 * np.log(2)))
        y_l = CrystalExperiment._convert_w_to_l(CrystalExperiment._gaussian, ang_freq, w0, sig_w)
        y_l /= np.max(y_l)
        return beta + alpha * y_l

    def _load_image(self, filename):
        img = Image.open(f"processed/{filename}").convert("L")
        img = np.array(img, dtype=float)
        img /= np.max(img)
        return img

    def _get_crop(self, img):
        return img[self.config["top"]:self.config["bottom"], self.config["left"]:self.config["right"]]

    def _get_meta_string(self):
        """Красивая строка со всеми параметрами для графиков и логов"""
        if not self.metadata:
            return "No Meta"
        return ", ".join([f"{k}: {v}" for k, v in self.metadata.items()])

    def run_calibration(self):
        img = self._load_image(self.filter_filename)
        crop = self._get_crop(img)
        x_pix = np.arange(self.config["left"], self.config["right"])
        y_exp = np.trapezoid(crop, axis=0)
        y_exp /= np.max(y_exp)

        p0 = [1.0, 0.1, 696.5, 0.37]
        popt, _ = curve_fit(self._gfit, x_pix, y_exp, p0=p0, maxfev=10000)
        y_fit = self._gfit(x_pix, *popt)
        
        # Сохраняем результаты калибровки в явные поля класса
        self.alpha = popt[0]
        self.beta = popt[1]
        self.shift = popt[2]
        self.scale = popt[3]
        self.r2 = r2_score(y_exp, y_fit)
        
        # Временный контекст для отрисовки локального графика калибровки
        self._tmp_calib = {"image": img, "crop": crop, "x_pix": x_pix, "y_exp": y_exp, "y_fit": y_fit}

    def process_nofilter_data(self):
        if self.shift is None or self.scale is None:
            raise ValueError("Калибровка не проведена или завершилась с ошибкой!")

        self.img_nf = self._load_image(self.nofilter_filename)
        self.crop_nf = self._get_crop(self.img_nf)
        self.spectrum_nf = np.trapezoid(self.crop_nf, axis=0)

        self.pixel808 = (self.config["target_wavelength"] - self.shift) / self.scale
        pixel_local = int(round(self.pixel808)) - self.config["left"]

        # 1. Метрики интенсивности из горизонтального спектра
        if 0 <= pixel_local < len(self.spectrum_nf):
            self.intensity_808_exact = self.spectrum_nf[pixel_local]
        else:
            self.intensity_808_exact = 0.0

        radius = 2
        w_start = max(0, pixel_local - radius)
        w_end = min(len(self.spectrum_nf), pixel_local + radius + 1)
        self.intensity_808_avg = np.mean(self.spectrum_nf[w_start:w_end]) if w_start < w_end else 0.0

        self.intensity_total = np.trapezoid(self.spectrum_nf)

        # 2. Вертикальный срез интерференции (анализ по оси Y)
        half_width = self.config["stripe_width"] // 2
        start_x = max(0, pixel_local - half_width)
        stop_x = min(self.crop_nf.shape[1], pixel_local + half_width + 1)
        self.slice_808 = np.mean(self.crop_nf[:, start_x:stop_x], axis=1)

        w_len = self.config["savgol_window"]
        p_order = self.config["savgol_poly"]
        if w_len >= len(self.slice_808):
            w_len = len(self.slice_808) if len(self.slice_808) % 2 != 0 else len(self.slice_808) - 1
        self.smoothed_slice = savgol_filter(self.slice_808, window_length=w_len, polyorder=p_order)

        self.intensity_max_slice = np.max(self.smoothed_slice)

        # 3. Поиск экстремумов интерференции
        peaks, _ = find_peaks(
            self.smoothed_slice, 
            prominence=self.config["peak_prominence"], 
            distance=self.config["peak_distance"],
            height=self.config["peak_height_min"]
        )
        self.peaks_idx = peaks

        valleys = []
        if len(peaks) >= 2:
            p_left = peaks[0]
            p_right = peaks[-1]
            valley_zone = -self.smoothed_slice[p_left:p_right]
            valleys_local, _ = find_peaks(
                valley_zone, 
                prominence=self.config["valley_prominence"], 
                distance=self.config["peak_distance"]
            )
            valleys = valleys_local + p_left
        self.valleys_idx = valleys

        extrema_list = []
        for p in peaks: extrema_list.append((p, 'max', self.smoothed_slice[p]))
        for v in valleys: extrema_list.append((v, 'min', self.smoothed_slice[v]))
        extrema_list.sort(key=lambda x: x[0])

        self.local_visibilities = []
        self.paired_extrema = []
        for i in range(len(extrema_list) - 1):
            ex1, ex2 = extrema_list[i], extrema_list[i+1]
            if ex1[1] != ex2[1]:  
                i_max = ex1[2] if ex1[1] == 'max' else ex2[2]
                i_min = ex1[2] if ex1[1] == 'min' else ex2[2]
                v_local = (i_max - i_min) / (i_max + i_min)
                self.local_visibilities.append(v_local)
                self.paired_extrema.append((ex1[0], ex2[0], ex1[1], ex2[1]))

        self.visibility = np.max(self.local_visibilities) if len(self.local_visibilities) > 0 else 0.0

        # Сборка лога с учетом метаданных
        self.log_line = (
            f"Params [{self._get_meta_string()}] | "
            f"Vis_Max={self.visibility:.4f} | "
            f"Exact_808={self.intensity_808_exact:.3f} | "
            f"Avg_808={self.intensity_808_avg:.3f}"
        )

    def save_plots(self):
        os.makedirs(self.filter_dir, exist_ok=True)
        os.makedirs(self.nofilter_dir, exist_ok=True)
        self._save_filter_plot()    
        self._save_nofilter_plot()

    def _save_filter_plot(self):
        if not hasattr(self, '_tmp_calib'): return
        res = self._tmp_calib
        info = (
            f"File: {os.path.basename(self.filter_filename)}\n"
            f"Meta: {self._get_meta_string()}\n\n"
            f"alpha = {self.alpha:.4f}\n"
            f"beta  = {self.beta:.4f}\n"
            f"shift = {self.shift:.4f} nm\n"
            f"scale = {self.scale:.6f} nm/pix\n"
            f"R²    = {self.r2:.5f}"
        )

        fig, ax = plt.subplots(2, 2, figsize=(10, 7))
        im0 = ax[0, 0].imshow(res["image"], aspect="auto")
        ax[0, 0].vlines([self.config["left"], self.config["right"]], 0, res["image"].shape[0])
        ax[0, 0].hlines([self.config["top"], self.config["bottom"]], 0, res["image"].shape[1])
        fig.colorbar(im0, ax=ax[0, 0])

        im1 = ax[0, 1].imshow(
            res["crop"],
            extent=[self.config["left"], self.config["right"], self.config["bottom"], self.config["top"]],
            aspect="auto",
        )
        fig.colorbar(im1, ax=ax[0, 1])

        ax[1, 0].plot(res["x_pix"], res["y_exp"], ".", label="Experiment")
        ax[1, 0].plot(res["x_pix"], res["y_fit"], lw=2, label="Fit")
        ax[1, 0].grid()
        ax[1, 0].legend()

        ax[1, 1].axis("off")
        ax[1, 1].text(0.05, 0.95, info, va="top", family="monospace")

        plt.tight_layout()
        out_name = os.path.basename(self.filter_filename).replace(".png", "_fit.png")
        plt.savefig(os.path.join(self.filter_dir, out_name), dpi=300)
        plt.close()
        del self._tmp_calib # Очищаем временный тяжелый массив

    def _save_nofilter_plot(self):
        fig, ax = plt.subplots(2, 2, figsize=(12, 8))

        im0 = ax[0, 0].imshow(
            self.crop_nf,
            extent=[self.config["left"], self.config["right"], self.config["bottom"], self.config["top"]],
            aspect="auto"
        )
        ax[0, 0].axvline(self.pixel808, color="red", linestyle="--", lw=1.5, label="808 nm")
        ax[0, 0].set_title(f"Crop Specs ({self._get_meta_string()})")
        ax[0, 0].legend()
        fig.colorbar(im0, ax=ax[0, 0])

        y_pix = np.arange(self.config["top"], self.config["bottom"])
        ax[0, 1].plot(y_pix, self.slice_808, color="purple", alpha=0.25, linestyle="--", label="Raw")
        ax[0, 1].plot(y_pix, self.smoothed_slice, color="purple", lw=2, label="Savitzky-Golay")
        
        y_peaks = self.config["top"] + self.peaks_idx
        y_valleys = self.config["top"] + self.valleys_idx
        
        ax[0, 1].plot(y_peaks, self.smoothed_slice[self.peaks_idx], "r^", markersize=7, label="Local Max")
        ax[0, 1].plot(y_valleys, self.smoothed_slice[self.valleys_idx], "bv", markersize=7, label="Bounded Min")

        if len(self.peaks_idx) >= 2:
            ax[0, 1].axvspan(self.config["top"] + self.peaks_idx[0], 
                             self.config["top"] + self.peaks_idx[-1], 
                             color='gainsboro', alpha=0.4, label='Valid Interference Zone')

        for idx1, idx2, _, _ in self.paired_extrema:
            y1 = self.config["top"] + idx1
            y2 = self.config["top"] + idx2
            ax[0, 1].plot([y1, y2], [self.smoothed_slice[idx1], self.smoothed_slice[idx2]], 
                         color="orange", linestyle="-", lw=1.2, alpha=0.8)

        ax[0, 1].set_title("Strict Bounded Extrema Matching")
        ax[0, 1].set_xlabel("Y coordinate (pixels)")
        ax[0, 1].set_ylabel("Intensity")
        ax[0, 1].grid(True, linestyle=":")
        ax[0, 1].legend()

        x_pix = np.arange(self.config["left"], self.config["right"])
        ax[1, 0].plot(x_pix, self.spectrum_nf, color="gray")
        ax[1, 0].axvline(self.pixel808, color="red", linestyle="--")
        ax[1, 0].set_title("Horizontal Spectrum Profile")
        ax[1, 0].grid(True)

        ax[1, 1].axis("off")
        info_text = (
            f"Parameters:\n {self._get_meta_string()}\n\n"
            f"Extrema Count:\n"
            f"  Max (peaks): {len(self.peaks_idx)}\n"
            f"  Min (bounded): {len(self.valleys_idx)}\n\n"
            f"Peak Intensity: {self.intensity_max_slice:.4f}\n"
            f"MAX VISIBILITY: V = {self.visibility:.5f}"
        )
        ax[1, 1].text(0.1, 0.8, info_text, va="top", family="monospace", fontsize=12,
                      bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.5))

        plt.tight_layout()
        out_name = os.path.basename(self.nofilter_filename).replace(".png", "_visibility.png")
        plt.savefig(os.path.join(self.nofilter_dir, out_name), dpi=200)
        plt.close()


class SpectroscopyBatchAnalyzer:
    """
    Менеджер автоматической пакетной обработки серий экспериментов.
    Умеет строить финальные зависимости по любому выбранному параметру метаданных.
    """
    def __init__(self, output_dir="analysis_output", x_param="distance"):
        self.output_dir = output_dir
        self.x_param = x_param  # Имя параметра из metadata, который будет осью X на финальных графиках
        self.experiments = []

    def add_experiment(self, exp: CrystalExperiment):
        self.experiments.append(exp)

    def process_all(self, generate_summary=False):
        os.makedirs(self.output_dir, exist_ok=True)
        for exp in self.experiments:
            print("-" * 60)
            exp.run_calibration()
            exp.process_nofilter_data()
            exp.save_plots()
            print(exp.log_line)
        if generate_summary:
            self._generate_summary_report()

    def plot_custom_report(self, x_param, y_param, group_by=None, 
                           title="Custom Plot", xlabel=None, ylabel=None, filename="custom_report.png"):
        """
        Универсальный метод для построения графиков строго точками.
        
        :param x_param: Ключ из metadata для оси X (например, 'slit_size')
        :param y_param: Имя поля класса CrystalExperiment для оси Y (например, 'visibility')
        :param group_by: Ключ из metadata для разделения на разные серии/цвета (например, 'distance')
        :param title: Заголовок графика
        :param xlabel: Подпись оси X (если None, берется имя параметра)
        :param ylabel: Подпись оси Y (если None, берется имя параметра)
        :param filename: Имя сохраняемого файла
        """
        plt.figure(figsize=(8, 5.5))
        
        # Список доступных маркеров для серий данных
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
        
        if group_by:
            # Находим все уникальные значения параметра группировки
            group_values = sorted(list(set([exp.metadata.get(group_by) for exp in self.experiments if group_by in exp.metadata])))
            
            for i, g_val in enumerate(group_values):
                # Фильтруем эксперименты, относящиеся к текущей группе
                sub_exps = [e for e in self.experiments if e.metadata.get(group_by) == g_val]
                # Сортируем внутри группы по оси X
                sub_exps.sort(key=lambda e: e.metadata.get(x_param, 0.0))
                
                x_data = [e.metadata.get(x_param) for e in sub_exps]
                y_data = [getattr(e, y_param) for e in sub_exps]
                
                marker = markers[i % len(markers)]
                plt.plot(x_data, y_data, marker, markersize=8, label=f"{group_by}: {g_val}")
        else:
            # Без группировки — просто один массив точек
            self.experiments.sort(key=lambda e: e.metadata.get(x_param, 0.0))
            x_data = [e.metadata.get(x_param) for e in self.experiments]
            y_data = [getattr(e, y_param) for e in self.experiments]
            plt.plot(x_data, y_data, 'o', color='royalblue', markersize=8)

        # Кастомизация осей и сетки
        plt.title(title, fontsize=12, fontweight="bold", pad=15)
        plt.xlabel(xlabel if xlabel else x_param, fontsize=11)
        plt.ylabel(ylabel if ylabel else y_param, fontsize=11)
        plt.grid(True, linestyle=":", alpha=0.6)
        
        if group_by:
            plt.legend(frameon=True, facecolor='white', edgecolor='gainsboro')
            
        plt.tight_layout()
        save_path = os.path.join(self.output_dir, filename)
        plt.savefig(save_path, dpi=300)
        plt.close()
        print(f"[График сохранен] -> {save_path}")

    def _generate_summary_report(self):
        if not self.experiments:
            print("Нет данных для финального отчета.")
            return

        # Сортируем эксперименты по значению выбранного X-параметра (например, по дистанции)
        self.experiments.sort(key=lambda x: x.metadata.get(self.x_param, 0.0))

        # Собираем массивы для графиков напрямую из полей классов
        x_values = np.array([exp.metadata.get(self.x_param, 0.0) for exp in self.experiments])
        visibilities = np.array([exp.visibility for exp in self.experiments])
        intensities_exact = np.array([exp.intensity_808_exact for exp in self.experiments])
        intensities_avg = np.array([exp.intensity_808_avg for exp in self.experiments])
        intensities_total = np.array([exp.intensity_total for exp in self.experiments])

        x_label = f"Experimental Parameter: '{self.x_param}'"

        # ---- ГРАФИК №1: МАКСИМАЛЬНАЯ ВИДНОСТЬ ----
        plt.figure(figsize=(7, 4.5))
        plt.plot(x_values, visibilities, "o", color="forestgreen", markersize=8)
        plt.xlabel(x_label, fontsize=11)
        plt.ylabel("Maximum Local Visibility (V)", fontsize=11)
        plt.title(f"Maximum Visibility vs {self.x_param}", fontsize=12, fontweight="bold")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, f"report_visibility_by_{self.x_param}.png"), dpi=300)
        plt.close()

        # ---- ГРАФИК №2: СТРОГАЯ ИНТЕНСИВНОСТЬ 808 НМ ----
        plt.figure(figsize=(7, 4.5))
        plt.plot(x_values, intensities_exact, "s", color="crimson", markersize=8)
        plt.xlabel(x_label, fontsize=11)
        plt.ylabel("Exact Intensity at 808 nm (a.u.)", fontsize=11)
        plt.title(f"Exact 808 nm Intensity vs {self.x_param}", fontsize=12, fontweight="bold")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, f"report_intensity_exact_by_{self.x_param}.png"), dpi=300)
        plt.close()

        # ---- ГРАФИК №3: УСРЕДНЕННАЯ ИНТЕНСИВНОСТЬ 808 НМ ----
        plt.figure(figsize=(7, 4.5))
        plt.plot(x_values, intensities_avg, "^", color="orangered", markersize=8)
        plt.xlabel(x_label, fontsize=11)
        plt.ylabel("Averaged Intensity at 808 nm (a.u.)", fontsize=11)
        plt.title(f"Averaged 808 nm Intensity vs {self.x_param}", fontsize=12, fontweight="bold")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, f"report_intensity_averaged_by_{self.x_param}.png"), dpi=300)
        plt.close()

        # ---- ГРАФИК №4: ПОЛНАЯ ИНТЕГРАЛЬНАЯ ИНТЕНСИВНОСТЬ ----
        plt.figure(figsize=(7, 4.5))
        plt.plot(x_values, intensities_total, "D", color="royalblue", markersize=7)
        plt.xlabel(x_label, fontsize=11)
        plt.ylabel("Total Integral Intensity (a.u.)", fontsize=11)
        plt.title(f"Total Integrated Intensity vs {self.x_param}", fontsize=12, fontweight="bold")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, f"report_intensity_integral_by_{self.x_param}.png"), dpi=300)
        plt.close()

        print(f"\n[Успех] Все 4 сводных графика сохранены в '{self.output_dir}/' с привязкой к оси '{self.x_param}'")

    def to_dataframe(self) -> pd.DataFrame:
        """Собирает все метаданные и посчитанные поля в один плоский DataFrame."""
        rows = []
        for exp in self.experiments:
            # Базовые метаданные (разворачиваем словарь в ключи строки)
            row_data = dict(exp.metadata)
            
            # Добавляем вычисленные физические параметры
            row_data.update({
                "visibility": exp.visibility,
                "intensity_808_exact": exp.intensity_808_exact,
                "intensity_808_avg": exp.intensity_808_avg,
                "intensity_total": exp.intensity_total,
                "intensity_max_slice": exp.intensity_max_slice,
                "calib_shift": exp.shift,
                "calib_scale": exp.scale,
                "calib_r2": exp.r2,
                "filter_file": exp.filter_filename,
                "nofilter_file": exp.nofilter_filename
            })
            rows.append(row_data)
            
        return pd.DataFrame(rows)
    



In [98]:
if __name__ == "__main__":
    # Демонстрация: задаем любые файлы и вешаем на них абсолютно любые метаданные
    experiments_data = [

        {
            "filter": "104_6_FILTER.png", 
            "nofilter": "104_6_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 6.0}
        },
        {
            "filter": "104_4_FILTER.png", 
            "nofilter": "104_4_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 4.0}
        },
        {
            "filter": "104_3_FILTER.png", 
            "nofilter": "104_3_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 3.0}
        },
        {
            "filter": "104_1_5_FILTER.png", 
            "nofilter": "104_1_5_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 1.5}
        },
        {
            "filter": "104_1_FILTER.png", 
            "nofilter": "104_1_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 1.0}
        },
        {
            "filter": "104_2_FILTER.png", 
            "nofilter": "104_2_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 2.0}
        },
        {
            "filter": "104_2_5_FILTER.png", 
            "nofilter": "104_2_5_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 2.5}
        },
        {
            "filter": "104_1_8_FILTER.png", 
            "nofilter": "104_1_8_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 1.8}
        },
                {
            "filter": "77_1_8_FILTER.png", 
            "nofilter": "77_1_8_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 1.8}
        },
        {
            "filter": "77_2_5_FILTER.png", 
            "nofilter": "77_2_5_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 2.5}
        },
        {
            "filter": "77_2_2_FILTER.png", 
            "nofilter": "77_2_2_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 2.2}
        },
        {
            "filter": "77_3_5_FILTER.png", 
            "nofilter": "77_3_5_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 3.5}
        },
        {
            "filter": "77_1_FILTER.png", 
            "nofilter": "77_1_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 1.0}
        },
        {
            "filter": "77_3_FILTER.png", 
            "nofilter": "77_3_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 3.0}
        },
        {
            "filter": "77_4_FILTER.png", 
            "nofilter": "77_4_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 4.0}
        },
        {
            "filter": "77_6_FILTER.png", 
            "nofilter": "77_6_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 6.0}
        },
                {
            "filter": "77_2_FILTER.png", 
            "nofilter": "77_2_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 2.0}
        },
                {
            "filter": "77_1_5_FILTER.png", 
            "nofilter": "77_1_5_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 1.5}
        }

    ]

    config_params = {
        "left": 250, "right": 370,
        "top": 140, "bottom": 275,
        "target_wavelength": 808.0,
        "stripe_width": 3,
        "savgol_window": 21,       
        "savgol_poly": 7,          
        "peak_prominence": 0.05,   
        "valley_prominence": 0.05, 
        "peak_distance": 6,        
        "peak_height_min": 0.12,   
        "output_dir": "analysis_output"
    }

    # Инициализируем анализатор серии. Указываем, по какому именно параметру строить финальные графики.
    # Например, хотим графики зависимости от дистанции:
    batch_manager = SpectroscopyBatchAnalyzer(output_dir=config_params["output_dir"], x_param="slit_size")
    
    # Если бы вы хотели отследить влияние размера щели, вы бы написали:
    # batch_manager = SpectroscopyBatchAnalyzer(output_dir=config_params["output_dir"], x_param="slit_size")

    for item in experiments_data:
        batch_manager.add_experiment(
            CrystalExperiment(
                filter_filename=item["filter"], 
                nofilter_filename=item["nofilter"], 
                metadata=item["meta"], 
                config=config_params
            )
        )

    batch_manager.process_all()


# 2. Получаем заветную табличку!
    df = batch_manager.to_dataframe()
    
    # Можно сразу вывести её в консоль для проверки:
    print("\nСгенерированный датафрейм:")
    print(df[["distance", "slit_size", "visibility", "intensity_808_exact"]].to_string())


------------------------------------------------------------
Params [distance: 104.0, slit_size: 6.0] | Vis_Max=0.3426 | Exact_808=48.783 | Avg_808=49.828
------------------------------------------------------------
Params [distance: 104.0, slit_size: 4.0] | Vis_Max=0.4092 | Exact_808=58.029 | Avg_808=56.829
------------------------------------------------------------
Params [distance: 104.0, slit_size: 3.0] | Vis_Max=0.4555 | Exact_808=57.578 | Avg_808=56.781
------------------------------------------------------------
Params [distance: 104.0, slit_size: 1.5] | Vis_Max=0.7965 | Exact_808=38.731 | Avg_808=38.054
------------------------------------------------------------
Params [distance: 104.0, slit_size: 1.0] | Vis_Max=0.6226 | Exact_808=19.942 | Avg_808=19.286
------------------------------------------------------------
Params [distance: 104.0, slit_size: 2.0] | Vis_Max=0.6209 | Exact_808=52.125 | Avg_808=51.589
------------------------------------------------------------
Params [d

In [101]:
def plot_parameter_vs_slit(df, y_param, distances, ylabel, title, filename=None, output_dir="analysis_output"):
    """
    Универсальная функция для построения зависимости любого параметра от размера щели
    для заданных дистанций с помощью чистого plt.
    
    :param df: Исходный pandas DataFrame с результатами
    :param y_param: Имя столбца для оси Y ('visibility', 'intensity_808_exact' и т.д.)
    :param distances: Список интересующих дистанций, например [77.0, 104.0]
    :param ylabel: Красивая подпись для оси Y
    :param title: Заголовок графика
    :param filename: Имя файла для сохранения (если None, график просто покажется на экран)
    :param output_dir: Папка для сохранения графиков
    """
    plt.figure(figsize=(7.5, 5.0))
    
    # Набор легко различимых маркеров для разных дистанций
    markers = ['o', 's', '^', 'D', 'v', 'p', '*']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'] # Стандартная контрастная палитра
    
    for i, dist in enumerate(distances):
        # 1. Фильтруем датафрейм по конкретной дистанции
        # Используем np.isclose на случай, если дистанция во float записана как 77.0001
        sub_df = df[df['distance'] == dist]
        
        if sub_df.empty:
            print(f"[Предупреждение] Нет данных для дистанции: {dist}")
            continue
            
        # 2. Обязательно сортируем по оси X (размер щели), чтобы точки шли по порядку
        sub_df = sub_df.sort_values(by='slit_size')
        
        # 3. Строим строго точками (linestyle='none')
        plt.plot(
            sub_df['slit_size'], 
            sub_df[y_param], 
            marker=markers[i % len(markers)], 
            color=colors[i % len(colors)],
            linestyle='none', 
            markersize=8, 
            markeredgecolor='black',  # Красивая темная обводка вокруг точек
            markeredgewidth=0.7,
            label=f"Distance: {dist} mm"
        )
        
    # Настройка осей и сетки
    plt.title(title, fontsize=12, fontweight="bold", pad=12)
    plt.xlabel("Slit Size (mm)", fontsize=11)
    plt.ylabel(ylabel, fontsize=11)
    plt.grid(True, linestyle=":", alpha=0.6, color="gray")
    plt.legend(frameon=True, facecolor='white', edgecolor='gainsboro', loc='best')
    
    plt.tight_layout()
    
    # Логика сохранения/вывода
    if filename:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, filename)
        plt.savefig(save_path, dpi=300)
        print(f"[Успех] График сохранен в {save_path}")
    else:
        plt.show()
        
    plt.close()


def plot_individual_per_distance(df, y_param, distances, ylabel, title_base, filename_prefix=None, output_dir="analysis_output"):
    """
    Создает отдельный график для каждой дистанции из списка.
    
    :param df: Исходный pandas DataFrame
    :param y_param: Столбец для оси Y ('visibility', 'intensity_808_exact' и т.д.)
    :param distances: Список дистанций, для каждой из которых нужен свой график
    :param ylabel: Подпись оси Y
    :param title_base: Базовая часть заголовка (дистанция добавится автоматически)
    :param filename_prefix: Префикс имени файла (например, 'visibility_dist')
    :param output_dir: Папка для сохранения
    """
    # Будем использовать один красивый цвет для индивидуальных графиков (например, глубокий синий)
    plot_color = '#1f77b4' 
    
    for dist in distances:
        # Фильтруем данные только для текущей дистанции
        sub_df = df[df['distance'] == dist]
        
        if sub_df.empty:
            print(f"[Пропущено] Нет данных для дистанции: {dist} mm")
            continue
            
        # Сортируем по размеру щели
        sub_df = sub_df.sort_values(by='slit_size')
        
        # Создаем новое отдельное окно/фигуру
        plt.figure(figsize=(6.5, 4.5))
        
        # Строим строго точками
        plt.plot(
            sub_df['slit_size'], 
            sub_df[y_param], 
            marker='o', 
            color=plot_color,
            linestyle='none', 
            markersize=8.5, 
            markeredgecolor='black', 
            markeredgewidth=0.8,
            label=f"Data points"
        )
        
        # Индивидуальные подписи с указанием конкретной дистанции
        plt.title(f"{title_base} (Distance: {dist} mm)", fontsize=12, fontweight="bold", pad=12)
        plt.xlabel("Slit Size (mm)", fontsize=11)
        plt.ylabel(ylabel, fontsize=11)
        plt.grid(True, linestyle=":", alpha=0.6, color="gray")
        plt.legend(frameon=True, facecolor='white', edgecolor='gainsboro')
        
        plt.tight_layout()
        
        # Сохранение с уникальным именем для каждой дистанции
        if filename_prefix:
            os.makedirs(output_dir, exist_ok=True)
            # Заменяем точки в названии файла, чтобы не ломать расширение (например, 77.0 -> 77)
            dist_str = str(dist).replace('.', '_')
            filename = f"{filename_prefix}_{dist_str}.png"
            save_path = os.path.join(output_dir, filename)
            plt.savefig(save_path, dpi=300)
            print(f"[Сохранено] -> {save_path}")
        else:
            plt.show()
            
        # Обязательно закрываем фигуру, чтобы не перегружать память
        plt.close()
# Задаем список дистанций, которые ищем в таблице
target_distances = [77.0, 104.0]

# 1. Сводный график для ВИДНОСТИ
plot_parameter_vs_slit(
    df=df,
    y_param="visibility",
    distances=target_distances,
    ylabel="Maximum Local Visibility (V)",
    title="Visibility vs Slit Size",
    filename="slit_vs_visibility_77_104.png"  # Уберите этот параметр, если нужно просто вывести на экран в Jupyter
)

# 2. Сводный график для ИНТЕНСИВНОСТИ (например, точной на 808 нм)
plot_parameter_vs_slit(
    df=df,
    y_param="intensity_808_exact",
    distances=target_distances,
    ylabel="Exact Intensity at 808 nm (a.u.)",
    title="Exact 808 nm Intensity vs Slit Size",
    filename="slit_vs_intensity_exact_77_104.png"
)

# 3. Сводный график для ИНТЕНСИВНОСТИ (усредненной в радиусе 2 пикселей)
plot_parameter_vs_slit(
    df=df,
    y_param="intensity_808_avg",
    distances=target_distances,
    ylabel="Averaged Intensity at 808 nm (a.u.)",
    title="Averaged 808 nm Intensity vs Slit Size",
    filename="slit_vs_intensity_avg_77_104.png"
)

[Успех] График сохранен в analysis_output\slit_vs_visibility_77_104.png
[Успех] График сохранен в analysis_output\slit_vs_intensity_exact_77_104.png
[Успех] График сохранен в analysis_output\slit_vs_intensity_avg_77_104.png


In [102]:
target_distances = [77.0, 104.0]

# 1. Получаем индивидуальные графики ВИДНОСТИ
plot_individual_per_distance(
    df=df,
    y_param="visibility",
    distances=target_distances,
    ylabel="Maximum Local Visibility (V)",
    title_base="Visibility vs Slit Size",
    filename_prefix="indiv_visibility"  # Создаст файлы: indiv_visibility_77_0.png и indiv_visibility_104_0.png
)

# 2. Получаем индивидуальные графики ИНТЕНСИВНОСТИ
plot_individual_per_distance(
    df=df,
    y_param="intensity_808_exact",
    distances=target_distances,
    ylabel="Exact Intensity at 808 nm (a.u.)",
    title_base="Exact 808 nm Intensity vs Slit Size",
    filename_prefix="indiv_intensity_exact" # Создаст файлы: indiv_intensity_exact_77_0.png и т.д.
)

[Сохранено] -> analysis_output\indiv_visibility_77_0.png
[Сохранено] -> analysis_output\indiv_visibility_104_0.png
[Сохранено] -> analysis_output\indiv_intensity_exact_77_0.png
[Сохранено] -> analysis_output\indiv_intensity_exact_104_0.png


In [90]:
if __name__ == "__main__":
    # Демонстрация: задаем любые файлы и вешаем на них абсолютно любые метаданные
    experiments_data = [

        {
            "filter": "104_6_FILTER.png", 
            "nofilter": "104_6_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 6.0}
        },
        {
            "filter": "104_4_FILTER.png", 
            "nofilter": "104_4_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 4.0}
        },
        {
            "filter": "104_3_FILTER.png", 
            "nofilter": "104_3_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 3.0}
        },
        {
            "filter": "104_1_5_FILTER.png", 
            "nofilter": "104_1_5_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 1.5}
        },
        {
            "filter": "104_1_FILTER.png", 
            "nofilter": "104_1_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 1.0}
        },
        {
            "filter": "104_2_FILTER.png", 
            "nofilter": "104_2_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 2.0}
        },
        {
            "filter": "104_2_5_FILTER.png", 
            "nofilter": "104_2_5_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 2.5}
        },
        {
            "filter": "104_1_8_FILTER.png", 
            "nofilter": "104_1_8_NOFILTER.png", 
            "meta": {"distance": 104.0, "slit_size": 1.8}
        }

    ]

    config_params = {
        "left": 250, "right": 370,
        "top": 140, "bottom": 275,
        "target_wavelength": 808.0,
        "stripe_width": 3,
        "savgol_window": 21,       
        "savgol_poly": 7,          
        "peak_prominence": 0.05,   
        "valley_prominence": 0.05, 
        "peak_distance": 6,        
        "peak_height_min": 0.12,   
        "output_dir": "analysis_output"
    }

    # Инициализируем анализатор серии. Указываем, по какому именно параметру строить финальные графики.
    # Например, хотим графики зависимости от дистанции:
    batch_manager = SpectroscopyBatchAnalyzer(output_dir=config_params["output_dir"], x_param="slit_size")
    
    # Если бы вы хотели отследить влияние размера щели, вы бы написали:
    # batch_manager = SpectroscopyBatchAnalyzer(output_dir=config_params["output_dir"], x_param="slit_size")

    for item in experiments_data:
        batch_manager.add_experiment(
            CrystalExperiment(
                filter_filename=item["filter"], 
                nofilter_filename=item["nofilter"], 
                metadata=item["meta"], 
                config=config_params
            )
        )

    batch_manager.process_all()


# 2. Получаем заветную табличку!
    df = batch_manager.to_dataframe()
    
    # Можно сразу вывести её в консоль для проверки:
    print("\nСгенерированный датафрейм:")
    print(df[["distance", "slit_size", "visibility", "intensity_808_exact"]].to_string())

    # =========================================================================
    # ТЕПЕРЬ СТРОИТЬ ГРАФИКИ СТАЛО ИГРУШКОЙ:
    # =========================================================================
    os.makedirs("analysis_output", exist_ok=True)
    sns.set_theme(style="whitegrid")

    # Пример 1: Видность от Размера щели, разбитая по Дистанциям (разные цвета и маркеры)
    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=df, 
        x="slit_size", 
        y="visibility", 
        hue="distance",      # Группировка цветом по дистанции
        style="distance",    # Группировка формой точек по дистанции
        palette="viridis",   # Красивая палитра цветов
        s=100                # Размер точек
    )
    plt.title("Visibility vs Slit Size (Grouped by Distance)", fontsize=12, fontweight="bold")
    plt.xlabel("Slit Size (mm)")
    plt.ylabel("Max Local Visibility (V)")
    plt.tight_layout()
    plt.savefig("analysis_output/pandas_visibility_report.png", dpi=300)
    plt.close()

    # Пример 2: Интенсивность от Дистанции, разбитая по Щелям
    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=df, 
        x="distance", 
        y="intensity_808_exact", 
        hue="slit_size", 
        palette="flare", 
        s=100
    )
    plt.title("Exact 808 nm Intensity vs Distance", fontsize=12, fontweight="bold")
    plt.xlabel("Crystal Separation (mm)")
    plt.ylabel("Intensity (a.u.)")
    plt.tight_layout()
    plt.savefig("analysis_output/pandas_intensity_report.png", dpi=300)
    plt.close()

------------------------------------------------------------
Params [distance: 104.0, slit_size: 6.0] | Vis_Max=0.3426 | Exact_808=48.783 | Avg_808=49.828
------------------------------------------------------------
Params [distance: 104.0, slit_size: 4.0] | Vis_Max=0.4092 | Exact_808=58.029 | Avg_808=56.829
------------------------------------------------------------
Params [distance: 104.0, slit_size: 3.0] | Vis_Max=0.4555 | Exact_808=57.578 | Avg_808=56.781
------------------------------------------------------------
Params [distance: 104.0, slit_size: 1.5] | Vis_Max=0.7965 | Exact_808=38.731 | Avg_808=38.054
------------------------------------------------------------
Params [distance: 104.0, slit_size: 1.0] | Vis_Max=0.6226 | Exact_808=19.942 | Avg_808=19.286
------------------------------------------------------------
Params [distance: 104.0, slit_size: 2.0] | Vis_Max=0.6209 | Exact_808=52.125 | Avg_808=51.589
------------------------------------------------------------
Params [d

In [91]:
if __name__ == "__main__":
    # Демонстрация: задаем любые файлы и вешаем на них абсолютно любые метаданные
    experiments_data = [

        {
            "filter": "77_1_8_FILTER.png", 
            "nofilter": "77_1_8_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 1.8}
        },
        {
            "filter": "77_2_5_FILTER.png", 
            "nofilter": "77_2_5_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 2.5}
        },
        {
            "filter": "77_2_2_FILTER.png", 
            "nofilter": "77_2_2_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 2.2}
        },
        {
            "filter": "77_3_5_FILTER.png", 
            "nofilter": "77_3_5_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 3.5}
        },
        {
            "filter": "77_1_FILTER.png", 
            "nofilter": "77_1_NOFILTER.png", 
            "meta": {"distance": 77.0, "slit_size": 1.0}
        },
        # {
        #     "filter": "77_3_FILTER.png", 
        #     "nofilter": "77_3_NOFILTER.png", 
        #     "meta": {"distance": 77.0, "slit_size": 3.0}
        # },
        # {
        #     "filter": "77_4_FILTER.png", 
        #     "nofilter": "77_4_NOFILTER.png", 
        #     "meta": {"distance": 77.0, "slit_size": 4.0}
        # },
        # {
        #     "filter": "77_6_FILTER.png", 
        #     "nofilter": "77_6_NOFILTER.png", 
        #     "meta": {"distance": 77.0, "slit_size": 6.0}
        # },
        #         {
        #     "filter": "77_2_FILTER.png", 
        #     "nofilter": "77_2_NOFILTER.png", 
        #     "meta": {"distance": 77.0, "slit_size": 2.0}
        # },
        #         {
        #     "filter": "77_1_5_FILTER.png", 
        #     "nofilter": "77_1_5_NOFILTER.png", 
        #     "meta": {"distance": 77.0, "slit_size": 1.5}
        # }

    ]

    config_params = {
        "left": 250, "right": 370,
        "top": 140, "bottom": 275,
        "target_wavelength": 808.0,
        "stripe_width": 3,
        "savgol_window": 21,       
        "savgol_poly": 7,          
        "peak_prominence": 0.05,   
        "valley_prominence": 0.05, 
        "peak_distance": 6,        
        "peak_height_min": 0.12,   
        "output_dir": "analysis_output"
    }

    # Инициализируем анализатор серии. Указываем, по какому именно параметру строить финальные графики.
    # Например, хотим графики зависимости от дистанции:
    batch_manager = SpectroscopyBatchAnalyzer(output_dir=config_params["output_dir"], x_param="slit_size")
    
    # Если бы вы хотели отследить влияние размера щели, вы бы написали:
    # batch_manager = SpectroscopyBatchAnalyzer(output_dir=config_params["output_dir"], x_param="slit_size")

    for item in experiments_data:
        batch_manager.add_experiment(
            CrystalExperiment(
                filter_filename=item["filter"], 
                nofilter_filename=item["nofilter"], 
                metadata=item["meta"], 
                config=config_params
            )
        )

    batch_manager.process_all()

------------------------------------------------------------
Params [distance: 77.0, slit_size: 1.8] | Vis_Max=0.6415 | Exact_808=25.789 | Avg_808=25.751
------------------------------------------------------------
Params [distance: 77.0, slit_size: 2.5] | Vis_Max=0.4856 | Exact_808=42.372 | Avg_808=41.523
------------------------------------------------------------
Params [distance: 77.0, slit_size: 2.2] | Vis_Max=0.5570 | Exact_808=37.814 | Avg_808=38.886
------------------------------------------------------------
Params [distance: 77.0, slit_size: 3.5] | Vis_Max=0.4122 | Exact_808=55.724 | Avg_808=55.089
------------------------------------------------------------
Params [distance: 77.0, slit_size: 1.0] | Vis_Max=0.7034 | Exact_808=10.466 | Avg_808=9.810
